# E7 Notebook 26 — Full CIC-IDS2017 source audit

Read-only full scan of eight publisher-style CIC-IDS2017 TrafficLabelling CSVs, with private research source unaffected.

- Real Kaggle source root: /kaggle/input/datasets/dsshayan/dscicids/
- Confirmed directory name: TrafficLabelling [trailing space]
- Confirmed timestamp header:  Timestamp [leading space]
- First example: 7/7/2017 3:30
- Experiment ID: E7-EXT-VALIDITY-001-G0

No model training, no UNSW official test access and no causal-ordering claims. Each **completed** public aggregate audit is automatically uploaded and verified in the dedicated E7 GitHub research folder. Requires Kaggle Internet and the existing pushRare secret.

In [ ]:
from pathlib import Path
import subprocess, sys

REPO=Path('/kaggle/working/rare-attack-e7-cicids')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1',
        'https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git',
        str(REPO)],check=True)
else:
    assert (REPO/'.git').is_dir(),'Not a Git checkout: refuse replacement.'
    changed=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip()
    assert not changed,'Uncommitted E7 checkout changes: refuse overwrite.'
    subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin','main'],check=True)
    subprocess.run(['git','-C',str(REPO),'merge','--ff-only','FETCH_HEAD'],check=True)

STUDY=REPO/'research/E7_CICIDS2017'
assert (STUDY/'scripts/e7_g0_external_source_audit.py').is_file()
assert (STUDY/'scripts/e7_g0_publish.py').is_file()
print('FROZEN E7 GITHUB SOURCE COMMIT:',
      subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

subprocess.run([sys.executable,'-m','pytest','-q',
    str(STUDY/'tests/test_e7_g0_external_source_audit.py'),
    str(STUDY/'tests/test_e7_g0_publish.py')],check=True)
print('E7 SOURCE AND UPLOAD TESTS PASSED')

## Step 2 — Confirm all eight CSVs and timestamp sample formats

The timestamp format is explicitly selected from your Friday sample. Each other file must pass its own preview check. We display **timestamp strings only**, never network endpoints or raw flow records.

In [ ]:
from pathlib import Path
import pandas as pd

EXTERNAL_ROOT=Path('/kaggle/input/datasets/dsshayan/dscicids')
assert EXTERNAL_ROOT.is_dir(),'Attach the dscicids dataset to Kaggle.'
FILES=sorted(EXTERNAL_ROOT.rglob('*.csv'))
assert len(FILES)==8,f'Expected eight source CSV files; found {len(FILES)}.'
assert len({str(f.parent) for f in FILES})==1,'Unexpected source folder layout.'
assert FILES[0].parent.name=='TrafficLabelling ','Unexpected source folder name.'
print('ACTUAL SOURCE DIRECTORY:',repr(str(FILES[0].parent)))

DATE_FORMAT='%d/%m/%Y %H:%M'
for number,p in enumerate(FILES,1):
    preview=pd.read_csv(p,nrows=3,encoding='utf-8-sig',
        usecols=lambda column:column.strip()=='Timestamp')
    assert preview.shape==(3,1),f'{p.name}: Timestamp missing.'
    raw_times=preview.iloc[:,0].astype(str).tolist()
    parsed=pd.to_datetime(preview.iloc[:,0],format=DATE_FORMAT,
        errors='coerce',exact=True)
    if parsed.isna().any():
        raise ValueError(f'Timestamp format differs in {p.name}; stop for review.')
    print(f'[{number}/8] {p.name}: {raw_times}')

print('E7 ALL EIGHT TIMESTAMP PREVIEWS PARSED')
print('NOT proof of source chronology or completed-flow causality.')

## Step 3 — Source-wide G0 scan + automatic GitHub archive

This cell may take several minutes for eight large CSV files. It computes original per-CSV SHA256 digests and scans all rows in batches; nothing is trained. It requires the Kaggle secret pushRare, verifies GitHub access **before** scanning, writes a local checkpoint, and confirms the remote GitHub report before starting the next CSV.

A failed data-quality report is also uploaded (explicitly marked G0_NOT_CLEARED) and further work stops. Re-running verifies existing evidence rather than silently overwriting it.

In [ ]:
import json
import importlib.util
from kaggle_secrets import UserSecretsClient

def load_local_module(alias,script_path):
    spec=importlib.util.spec_from_file_location(alias,script_path)
    module=importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

audit=load_local_module('e7_source_audit',STUDY/'scripts/e7_g0_external_source_audit.py')
publisher=load_local_module('e7_public_sync',STUDY/'scripts/e7_g0_publish.py')
token=UserSecretsClient().get_secret('pushRare')
assert token,'GitHub pushRare secret required BEFORE full scans.'
publisher.github_json('GET',
    'https://api.github.com/repos/itsCodeBakery/rare-attack-spatiotemporal-ids',token)
print('GITHUB CONNECTED. Secret not displayed.')

OUT=Path('/kaggle/working/E7-EXT-VALIDITY-001/G0')
OUT.mkdir(parents=True,exist_ok=True)
completed=[]
for number,p in enumerate(FILES,1):
    print()
    print(f'E7 G0 [{number}/8] AUDITING: {p.name}',flush=True)
    report_path=OUT/(p.name+'.audit.json')
    if report_path.exists():
        report=json.loads(report_path.read_text(encoding='utf-8'))
        assert report['external_source_basename']==p.name,'Prior filename mismatch'
        assert report['input_artifact_sha256']==audit.fingerprint(p),'Prior SHA mismatch'
        assert report.get('full_file_scan') is True,'Prior checkpoint is sample-only'
        print('Original local checkpoint validated.',flush=True)
    else:
        report=audit.inspect(p,timestamp_format=DATE_FORMAT,
            max_rows=0,chunk_size=40000)
        temp=report_path.with_suffix(report_path.suffix+'.tmp')
        temp.write_text(json.dumps(report,indent=2,
            sort_keys=True,ensure_ascii=False)+'\n',encoding='utf-8')
        temp.replace(report_path)
        print('Read-only local source report preserved.',flush=True)
    receipt=publisher.publish_report(report_path,token=token,
        run_path=publisher.PREFIX)
    assert receipt['status'] in ('UPLOADED_AND_VERIFIED','ALREADY_SAVED_IDENTICALLY')
    print('GITHUB VERIFIED:',receipt['status'],flush=True)
    print('PUBLIC REPORT: https://github.com/'+publisher.REPO+
        '/blob/main/'+receipt['remote'],flush=True)
    stats=report['files'][0]
    print('Rows:',stats['rows_inspected'],
        '| Labels:',len(stats['labels']),
        '| Time parsing failures:',stats['parse_failures'],
        '| Reversed adjacent timestamps:',stats['timestamp_disorder_adjacent'],flush=True)
    completed.append(p.name)
    if report['status']!='G0_SCHEMA_CHECKED_TEMPORAL_VALIDITY_PENDING':
        raise RuntimeError('G0_NOT_CLEARED. Original failure preserved on GitHub; stop.')

print()
print('='*70)
print('ALL EIGHT E7 SOURCE AUDITS SAVED AND VERIFIED ON GITHUB')
print('https://github.com/'+publisher.REPO+'/tree/main/'+publisher.PREFIX)
print('NO MODEL TRAINING | NO UNSW TEST | FLOW-TIME VALIDITY STILL PENDING')
print('Send this output to ChatGPT before next research notebook.')
print('='*70)

## Stop after Notebook 26

Even a successful G0 data/schema scan cannot prove chronology or flow completion. Notebook 27 will separately check timestamp granularity, duration units and flow-end calculations, dataset label quality, duplicate source flows and host/session relationships before we register new chronological splits.

The E7 folder is independent of the frozen M1, E6 and original one-time UNSW official test results. No external classifier scores exist yet.